In [4]:
%pip install -q "transformers<5" langchain langchain-community \
                langchain-huggingface langchain-classic langchain-text-splitters \
                sentence-transformers faiss-cpu datasets

Note: you may need to restart the kernel to use updated packages.


  You can safely remove it manually.

[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
import os, platform, time, re
import numpy as np
import warnings
# Silence one known-harmless notice, not everything. On a day about silent
# failure, a blanket filter would hide the warnings you most want to see.
warnings.filterwarnings("ignore", message=".*resume_download.*")

if platform.system() == "Darwin":
    os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

from datasets import load_dataset
from sentence_transformers import SentenceTransformer
import transformers
transformers.logging.set_verbosity_error()
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, pipeline

from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings, HuggingFacePipeline
from langchain_community.vectorstores import FAISS
from langchain_classic.chains import RetrievalQA

print("Setup complete!")

c:\Users\avigd\Desktop\GENAI\.venv2\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


C:\Users\avigd\AppData\Local\Temp\ipykernel_29852\2687791330.py:20: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


Setup complete!


In [2]:
pages = load_dataset("m-ric/huggingface_doc", split="train[:200]")
texts, sources = list(pages["text"]), list(pages["source"])

lengths = np.array([len(t) for t in texts])
print(f"{len(texts)} documents")
print(f"characters: mean {lengths.mean():,.0f}   median {np.median(lengths):,.0f}   "
      f"longest {lengths.max():,}")
print(f"\nThe first 200 characters of document [3]:")
print(f"  {texts[3][:200]!r}")

200 documents
characters: mean 10,226   median 4,915   longest 371,057

The first 200 characters of document [3]:
  '!--Copyright 2023 The HuggingFace Team. All rights reserved.\n\nLicensed under the Apache License, Version 2.0 (the "License"); you may not use this file except in compliance with\nthe License. You may o'


In [3]:
embedder = SentenceTransformer('all-MiniLM-L6-v2')
QUESTION = "Why does XLA recompile the model when input shapes keep changing?"

doc_vectors = embedder.encode(texts, normalize_embeddings=True, batch_size=32)
q_vector = embedder.encode([QUESTION], normalize_embeddings=True)
scores = (doc_vectors @ q_vector.T).ravel()

# We found the page that answers it beforehand, by searching the raw text.
# That's the yardstick, not the method.
ANSWER_DOC = next(i for i, t in enumerate(texts) if "recompile your model for every" in t)
order = np.argsort(scores)[::-1]

print(f"QUESTION: {QUESTION}\n")
print("TOP 3 DOCUMENTS:")
for i in order[:3]:
    print(f"   {scores[i]:.3f}  {sources[i].split('/')[-1]}")

rank = list(order).index(ANSWER_DOC) + 1
print(f"\nThe page that answers it, {sources[ANSWER_DOC].split('/')[-1]},"
      f" scored {scores[ANSWER_DOC]:.3f}")
print(f"-> ranked {rank} out of {len(texts)}.")

QUESTION: Why does XLA recompile the model when input shapes keep changing?

TOP 3 DOCUMENTS:
   0.320  DESCRIPTION.md
   0.315  archived_changes.md
   0.300  dummy_input_generators.mdx

The page that answers it, perf_train_tpu_tf.md, scored 0.194
-> ranked 38 out of 200.


In [4]:
WINDOW = embedder.max_seq_length
token_counts = np.array([len(embedder.tokenizer.encode(t)) for t in texts])

print(f"Reading window: {WINDOW} tokens (~{WINDOW*4:,} characters)\n")
print(f"documents longer than that: {(token_counts > WINDOW).sum()} of {len(texts)}")
print(f"tokens in the corpus      : {token_counts.sum():,}")
print(f"tokens it actually read   : {np.minimum(token_counts, WINDOW).sum():,}")
print(f"\n-> {1 - np.minimum(token_counts, WINDOW).sum()/token_counts.sum():.0%} "
      f"of this corpus never reached the model.")
print(f"\nWhat it DID read from the page that answers our question:")
print(f"  {texts[ANSWER_DOC][:140]!r}")

Reading window: 256 tokens (~1,024 characters)

documents longer than that: 181 of 200
tokens in the corpus      : 680,393
tokens it actually read   : 48,786

-> 93% of this corpus never reached the model.

What it DID read from the page that answers our question:
  '!--Copyright 2023 The HuggingFace Team. All rights reserved.\n\nLicensed under the Apache License, Version 2.0 (the "License"); you may not us'


In [5]:
splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=150)
print("Separators it tries, in order:", splitter._separators)

chunk_texts, chunk_source = [], []
for i, t in enumerate(texts):
    for piece in splitter.split_text(t):
        chunk_texts.append(piece)
        chunk_source.append(i)

print(f"\n{len(texts)} documents -> {len(chunk_texts):,} chunks\n")

chunk_vectors = embedder.encode(chunk_texts, normalize_embeddings=True, batch_size=256)
chunk_scores = (chunk_vectors @ q_vector.T).ravel()

print("TOP 3 CHUNKS:")
for j in np.argsort(chunk_scores)[::-1][:3]:
    print(f"   {chunk_scores[j]:.3f}  from {sources[chunk_source[j]].split('/')[-1]}")

print(f"\n-> whole documents: rank {rank} of {len(texts)}, score {scores[ANSWER_DOC]:.3f}")
print(f"-> chunks:          rank 1,          score {chunk_scores.max():.3f}")

Separators it tries, in order: ['\n\n', '\n', ' ', '']

200 documents -> 2,784 chunks

TOP 3 CHUNKS:
   0.668  from perf_train_tpu_tf.md
   0.480  from perf_train_tpu_tf.md
   0.460  from perf_train_tpu_tf.md

-> whole documents: rank 38 of 200, score 0.194
-> chunks:          rank 1,          score 0.668


In [6]:
print(f"{'chunk_size':>11} {'chunks':>9} {'avg tokens':>12}   fits the 256 window?")
print("=" * 58)
for size, overlap in [(500, 50), (1000, 150), (2000, 200)]:
    sp = RecursiveCharacterTextSplitter(chunk_size=size, chunk_overlap=overlap)
    pieces = [p for t in texts for p in sp.split_text(t)]
    avg = np.mean([len(embedder.tokenizer.encode(p)) for p in pieces])
    print(f"{size:>11} {len(pieces):>9,} {avg:>12.0f}   "
          f"{'yes' if avg < 256 else 'NO - truncated again'}")

 chunk_size    chunks   avg tokens   fits the 256 window?
        500     5,966          118   yes
       1000     2,784          258   NO - truncated again
       2000     1,317          535   NO - truncated again


In [7]:
documents = [Document(page_content=t, metadata={"source": s})
             for t, s in zip(texts, sources)]
chunks = splitter.split_documents(documents)

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
db = FAISS.from_documents(chunks, embeddings)   # encodes AND indexes AND keeps the text

retriever = db.as_retriever(search_kwargs={"k": 4})
found = retriever.invoke(QUESTION)

print(f"FAISS store: {db.index.ntotal:,} vectors, {db.index.d} numbers each\n")
for d in found:
    print(f"   {d.metadata['source'].split('/')[-1]:26} {d.page_content[:58].strip()!r}...")

FAISS store: 2,784 vectors, 384 numbers each

   perf_train_tpu_tf.md       'Here, we avoid data-dependent shapes by computing the loss'...
   perf_train_tpu_tf.md       'How can you get around rule #3? The key is **padding** - i'...
   perf_train_tpu_tf.md       '```python\nlabel_mask = labels >= 0\nmasked_outputs = output'...
   perf_train_tpu_tf.md       '#### XLA Rule #2: Your code cannot have “data-dependent sh'...


In [8]:
LLM_ID = "google/flan-t5-small"
lm_tokenizer = AutoTokenizer.from_pretrained(LLM_ID)
lm_model = AutoModelForSeq2SeqLM.from_pretrained(LLM_ID)
generator = pipeline("text2text-generation", model=lm_model,
                     tokenizer=lm_tokenizer, max_new_tokens=80)
llm = HuggingFacePipeline(pipeline=generator)

qa = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=db.as_retriever(search_kwargs={"k": 2}),
    return_source_documents=True,
)

print(f"The window this model reports: {lm_tokenizer.model_max_length} tokens\n")
print("THE PROMPT LANGCHAIN SENDS, WHICH WE DID NOT WRITE:")
print("=" * 72)
print(qa.combine_documents_chain.llm_chain.prompt.template)
print("=" * 72)

The window this model reports: 512 tokens

THE PROMPT LANGCHAIN SENDS, WHICH WE DID NOT WRITE:
Use the following pieces of context to answer the question at the end. If you don't know the answer, just say that you don't know, don't try to make up an answer.

{context}

Question: {question}
Helpful Answer:


In [9]:
result = qa.invoke({"query": QUESTION})

print(f"QUESTION: {QUESTION}\n")
print(f"ANSWER:   {result['result']}\n")
print("BUILT FROM:")
for d in result["source_documents"]:
    print(f"   {d.metadata['source'].split('/')[-1]}")

QUESTION: Why does XLA recompile the model when input shapes keep changing?

ANSWER:   XLA will need to recompile your model for every different input shapes it sees

BUILT FROM:
   perf_train_tpu_tf.md
   perf_train_tpu_tf.md


In [10]:
def ask_and_show(question, k=2):
    out = RetrievalQA.from_chain_type(
        llm=llm, chain_type="stuff",
        retriever=db.as_retriever(search_kwargs={"k": k}),
        return_source_documents=True,
    ).invoke({"query": question})
    print(f"Q: {question}")
    print(f"A: {out['result'][:170]}")
    print("   retrieved from:")
    for d in out["source_documents"]:
        print(f"      {d.metadata['source'].split('/')[-1]}")
    print()
    return out

ask_and_show("How do I choose a metric for evaluation?")
ask_and_show("What is the Endpoint creation page?")

Q: How do I choose a metric for evaluation?
A: **Looking at the [Task pages](https://huggingface.co/tasks)** to see what metrics can be used for evaluating models for a given task. For example, see the [BLEU metric ca
   retrieved from:
      choosing_a_metric.mdx
      choosing_a_metric.mdx

Q: What is the Endpoint creation page?
A: [https://ui.endpoints.huggingface.co/new]
   retrieved from:
      mantis-case-study.md
      text-to-webapp.md



{'query': 'What is the Endpoint creation page?',
 'result': '[https://ui.endpoints.huggingface.co/new]',
 'source_documents': [Document(id='bd1a125d-e9ea-4dae-a6d2-97d5e5a09bba', metadata={'source': 'huggingface/blog/blob/main/mantis-case-study.md'}, page_content='## Other considerations\n\n### Deployment Options\n\nCurrently you can deploy an Inference Endpoint from the [GUI](https://ui.endpoints.huggingface.co/new) or using a [RESTful API](https://huggingface.co/docs/inference-endpoints/api_reference). You can also make use of our command line tool [hugie](https://github.com/MantisAI/hfie) (which will be the subject of a future blog) to launch Inference Endpoints in one line of code by passing a configuration, it’s really this simple:\n\n```bash\nhugie endpoint create example/development.json\n```'),
  Document(id='ac99d80c-1c1a-406e-9707-eb3b1e1c43f7', metadata={'source': 'huggingface/blog/blob/main/text-to-webapp.md'}, page_content='To deploy a new Endpoint you can go to the [Endpo